In [114]:
import os
from tqdm import tqdm
import numpy as np
import cv2
import random
# import bm3d import 
from bm3d import bm3d_deblurring, BM3DProfile, gaussian_kernel, gaussian_kernel
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [157]:
glioma = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/glioma_cropped/"
meningioma = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/meningioma_cropped_enhanced/"
meningioma_path = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train_enhanced/"
pituitary_path = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train_enhanced/"
pituitary = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/pituitary_tumor_cropped_enhanced/"
Training_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/"
Test_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/"
Val_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/"
augmented_set = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/oversampled"
paths = {'glioma': glioma, 'meningioma': meningioma, 'pituitary':pituitary}
print(paths)


{'glioma': 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/glioma_cropped/', 'meningioma': 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/meningioma_cropped_enhanced/', 'pituitary': 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/pituitary_tumor_cropped_enhanced/'}


In [163]:
# raw_cropped_image = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Training/G177_cropped.jpg"

def eualize_darken(image_to_crop_path):        
    
    img = cv2.imread(image_to_crop_path)    
    NOISE_FLOOR = 25

    def reduce_noise(noisy_img, noise_floor=NOISE_FLOOR):
        clean_img = np.array(noisy_img)
        clean_img[noisy_img < NOISE_FLOOR] = 0
        return clean_img

    new_img = reduce_noise(img)

    # Blurr
    new_img = cv2.bilateralFilter(new_img,3,20,20)

    img_hsv = cv2.cvtColor(new_img, cv2.COLOR_RGB2YCrCb)

    img_hsv[:, :, 0] = cv2.equalizeHist(img_hsv[:, :, 0])

    image = cv2.cvtColor(img_hsv, cv2.COLOR_YCrCb2RGB)    
    
    # res = np.hstack((new_img, image)) # this line stacks the images side by side to visualize the difference
    # cv2.imshow("equalize", image)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    
    # cv2.imwrite(
    #     "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Images/tests/equ_Dkn_01.jpg",
    #     image,
    # )
    # print("Saved!")
    return image

# equalized_darkened_img = eualize_darken(raw_cropped_image)

In [164]:
def gamma_correct(image, image_to_crop_path, cropped_images_folder):
        
    def gammaCorrection(src, gamma):
        invGamma = 1 / gamma

        table = [((i / 255) ** invGamma) * 255 for i in range(256)]
        table = np.array(table, np.uint8)

        return cv2.LUT(src, table)

    gamma_corrected_image = gammaCorrection(image, 1.4)
    
    # Saving
    current_image_name = image_to_crop_path.rsplit('/', 1)[1]
    # print("current_image_name: ", current_image_name)
    Cropped_image_name = current_image_name.rsplit('.')[0] + '_enhanced' + "."+ current_image_name.split('.')[1]
    # print("Cropped_image_name: ", Cropped_image_name)
    print(
        f"Writing the enhanced image {Cropped_image_name} to {cropped_images_folder}... "
    )
    Cropped_image_path = f"{cropped_images_folder}/{Cropped_image_name}"
    print("Enhanced_image_path: ", Cropped_image_path)
    # plt.imshow(Cropped_image)
    saved_cropped_image = cv2.imwrite(Cropped_image_path, gamma_corrected_image)

    # cv2.imshow("Gamma corrected image", gamma_corrected_image)
    # cv2.imwrite(
    #     "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Enhanced_images/",
    #     gamma_corrected_image,
    # )
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    print("Saved!")
    
    return gamma_corrected_image

# gamma_corrected_image = gamma_correct(equalized_darkened_img)

# Enhancing Training set

In [ ]:
Root = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/"
# Training_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Training/"
Children_directories = next(os.walk(Training_Classes))[1]

print("Children_directories: ", Children_directories)
for i in range(len(Children_directories)):  
    specific_class_folder = os.path.join(Training_Classes,Children_directories[i])
    print("specific_class_folder: ", specific_class_folder)
    
    # Create a new specific_class_folder for the cropped images
    cropped_images_folder = Training_Classes.replace('Training', 'Enhanced_Images') + Children_directories[i] + '_enhanced'
    # print("cropped_images_folder: ", cropped_images_folder)
    
    if not os.path.exists(cropped_images_folder):
        os.makedirs(cropped_images_folder)
    for root, dirs, files in os.walk(specific_class_folder):
        for file in tqdm(range(len(files))):
            image_path =root+ '/' + files[file]
            # print("image_path: ", image_path)
            equalized_darkened_img = eualize_darken(image_path)
            gamma_corrected_image = gamma_correct(equalized_darkened_img, image_path, cropped_images_folder)
            


# Enhancing Validation Set

In [172]:
Root = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/"
# Training_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Training/"
Children_directories = next(os.walk(Val_Classes))[1]

print("Children_directories: ", Children_directories)
for i in range(len(Children_directories)):  
    specific_class_folder = os.path.join(Val_Classes,Children_directories[i])
    print("specific_class_folder: ", specific_class_folder)
    
    # Create a new specific_class_folder for the cropped images
    cropped_images_folder = Val_Classes.replace('Val', 'Enhanced_Images') + Children_directories[i] + '_enhanced'
    # print("cropped_images_folder: ", cropped_images_folder)
    
    if not os.path.exists(cropped_images_folder):
        os.makedirs(cropped_images_folder)
    for root, dirs, files in os.walk(specific_class_folder):
        for file in range(len(files)):
            image_path =root+ '/' + files[file]
            # print("image_path: ", image_path)
            equalized_darkened_img = eualize_darken(image_path)
            gamma_corrected_image = gamma_correct(equalized_darkened_img, image_path, cropped_images_folder)
            


Children_directories:  ['glioma_cropped', 'glioma_cropped_enhanced', 'meningioma_cropped', 'pituitary_tumor_cropped']
specific_class_folder:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/glioma_cropped
Writing the enhanced image G1000_cropped_enhanced.jpg to C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/glioma_cropped_enhanced... 
Enhanced_image_path:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/glioma_cropped_enhanced/G1000_cropped_enhanced.jpg
Saved!
Writing the enhanced image G1004_cropped_enhanced.jpg to C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/glioma_cropped_enhanced... 
Enhanced_image_path:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/glioma_crop

# Enhancing Testing Set

In [173]:
Root = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/"
# Training_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Training/"
Children_directories = next(os.walk(Test_Classes))[1]

print("Children_directories: ", Children_directories)
for i in range(len(Children_directories)):  
    specific_class_folder = os.path.join(Test_Classes,Children_directories[i])
    print("specific_class_folder: ", specific_class_folder)
    
    # Create a new specific_class_folder for the cropped images
    cropped_images_folder = Test_Classes.replace('Test', 'Enhanced_Images') + Children_directories[i] + '_enhanced'
    # print("cropped_images_folder: ", cropped_images_folder)
    
    if not os.path.exists(cropped_images_folder):
        os.makedirs(cropped_images_folder)
    for root, dirs, files in os.walk(specific_class_folder):
        for file in range(len(files)):
            image_path =root+ '/' + files[file]
            # print("image_path: ", image_path)
            equalized_darkened_img = eualize_darken(image_path)
            gamma_corrected_image = gamma_correct(equalized_darkened_img, image_path, cropped_images_folder)
            


Children_directories:  ['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped']
specific_class_folder:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/glioma_cropped
Writing the enhanced image G1005_cropped_enhanced.jpg to C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/glioma_cropped_enhanced... 
Enhanced_image_path:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/glioma_cropped_enhanced/G1005_cropped_enhanced.jpg
Saved!
Writing the enhanced image G1023_cropped_enhanced.jpg to C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/glioma_cropped_enhanced... 
Enhanced_image_path:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/glioma_cropped_enhanced/G1023_cro

In [166]:
# def generate_imgages(train_dir, batch_size):
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip = True,
    vertical_flip = True,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.2,
    rotation_range = 10,
    fill_mode= 'constant',
    cval= 0,
    zoom_range = [0.8, 1]
)

# augmented_train_data = train_datagen.flow_from_directory(
#     directory= train_dir,
#     save_to_dir= augmented_set,
#     # classes= ['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped'],
#     # class_mode= 'categorical',
#     batch_size= batch_size,
#     seed= 22,
#     shuffle= True,
# )
    
    
    # return augmented_train_data

In [167]:
# dir_path = r'E:\account'
# Iterate directory
class_samples = {}
for k, v in paths.items():
    if k == 'x':
        break
    else:
        count = 0
        for path in os.listdir(paths[k]):
            # check if current path is a file
            if os.path.isfile(os.path.join(paths[k], path)):
                count += 1
        class_samples[k] = count
        print('There are {} images in the {} class:'.format(count, k))
print(class_samples)

There are 998 images in the glioma class:
There are 495 images in the meningioma class:
There are 651 images in the pituitary class:
{'glioma': 998, 'meningioma': 495, 'pituitary': 651}


In [168]:
lst = list(class_samples.values())
max_s = max(lst)
print(max_s)

key_list = list(class_samples.keys())
val_list = list(class_samples.values())
 
position = val_list.index(max_s)
majority_class = key_list[position]
print("majority_class: ",majority_class)
class_samples.pop(majority_class, None)
print("class_samples: ",class_samples)



998
majority_class:  glioma
class_samples:  {'meningioma': 495, 'pituitary': 651}


# Classical Augmentation

In [ ]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip = True,
    vertical_flip = True,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.2,
    rotation_range = 10,
    fill_mode= 'constant',
    cval= 0,
    zoom_range = [0.8, 1]
)

In [170]:
generated_images = 0
# for k, v in class_samples.items():
#     if k == 'meningioma':
#         pass
target_size = max_s - class_samples['pituitary']

# selector = random.randint(0, target_size)
augmmentation_rate = random.randint(1, 2)
print("The target size is: ", target_size)
# print("The selected image is: ", selector)
print("The augmmentation_rate is: ", augmmentation_rate)

print("Generating Images...")
train_path = pituitary_path
print("path: ", train_path)
for i in train_datagen.flow_from_directory(
    directory= train_path,
    save_to_dir= augmented_set,
    # classes= ['glioma_cropped', 'meningioma_cropped', 'pituitary_tumor_cropped'],
    # class_mode= 'categorical',
    batch_size= augmmentation_rate,
    seed= 22,
    shuffle= True,
):
    generated_images += augmmentation_rate
    print("generated_images: ", generated_images)
    if generated_images > target_size-1:
        print("Done!")
        break
        

    

The target size is:  347
The augmmentation_rate is:  1
Generating Images...
path:  C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train_enhanced/
Found 651 images belonging to 1 classes.
generated_images:  1
generated_images:  2
generated_images:  3
generated_images:  4
generated_images:  5
generated_images:  6
generated_images:  7
generated_images:  8
generated_images:  9
generated_images:  10
generated_images:  11
generated_images:  12
generated_images:  13
generated_images:  14
generated_images:  15
generated_images:  16
generated_images:  17
generated_images:  18
generated_images:  19
generated_images:  20
generated_images:  21
generated_images:  22
generated_images:  23
generated_images:  24
generated_images:  25
generated_images:  26
generated_images:  27
generated_images:  28
generated_images:  29
generated_images:  30
generated_images:  31
generated_images:  32
generated_images:  33
generated_images:  34
generated_ima

In [107]:
# # img = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Training/G177_cropped.jpg"

# # raw_cropped_image = cv2.imread(gamma_corrected_image)
# raw_cropped_image = gamma_corrected_image
# # raw_cropped_image = cv2.cvtColor(raw_cropped_image, cv2.COLOR_BGR2GRAY)

# contrast = 1 # Contrast control ( 0 to 127)
# brightness = 0. # Brightness control (0-100)
# out = cv2.convertScaleAbs(raw_cropped_image, alpha=contrast)
# # out = cv2.addWeighted(raw_cropped_image, contrast)

# out = cv2.normalize(out, None, alpha=0,beta=255, norm_type=cv2.NORM_MINMAX)

# comp = np.hstack((raw_cropped_image, out)) 
# cv2.imshow("raw_cropped_image_denoised", comp)
# cv2.waitKey(0)
# cv2.destroyAllWindows()